# NeutrinoJEPA Training

Run cells top to bottom. Checkpoints sync to Google Drive as they're written.

Runs all 3 experiments, each via the repo's own scripts (this notebook does
not reimplement any training logic):

1. **NeutrinoPET**: `train/pretrain_mae.py` -> `train/probe.py` -> `train/finetune.py`
2. **NeutrinoJEPAPET**: `train/pretrain.py` -> `train/probe.py` -> `train/finetune.py`
3. **PET+heads from scratch**: `train/finetune.py` on `configs/scratch.yaml` directly

Requires a Kaggle API token in Colab secrets as `KAGGLE_USERNAME` / `KAGGLE_KEY`.


In [ ]:
REPO_URL = "https://github.com/UnnatPar/neutrinojepa.git"
REPO_DIR = "/content/neutrinojepa"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/neutrinojepa_checkpoints"


In [ ]:
# -- Install torch + PyG (before cloning; these don't depend on the repo) --
# Install order matters: graphnet's own dependency chain has no upper bound
# on torch and will silently upgrade it past the pin if installed afterward
# (verified by direct testing while building this repo) -- torch/PyG are
# installed LAST, after requirements.txt, to guarantee the pinned versions win.
import subprocess, os, sys, re, threading, shutil, glob, time

subprocess.run(
    ["pip", "install", "-q", "torch==2.3.0", "--index-url", "https://download.pytorch.org/whl/cu121"],
    check=True,
)
print("torch installed")


In [ ]:
# -- Clone repo --
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
print("Repo ready at", REPO_DIR)


In [ ]:
# -- Install the rest of the repo's dependencies, then re-pin torch + PyG last --
subprocess.run(
    ["pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", "--no-deps", "torch==2.3.0",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    check=True,
)
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", "--no-deps",
     "torch-geometric==2.5.0", "torch-cluster==1.6.3", "torch-scatter==2.1.2",
     "-f", "https://data.pyg.org/whl/torch-2.3.0+cu121.html"],
    check=True,
)
print("Dependencies installed")


In [ ]:
# -- Mount Google Drive --
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will sync to: {DRIVE_CHECKPOINT_DIR}")


In [ ]:
# -- Kaggle credentials + data download (via the repo's own script) --
from google.colab import userdata

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json, "w") as f:
    f.write('{"username": "%s", "key": "%s"}' % (
        userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY"),
    ))
os.chmod(kaggle_json, 0o600)
subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

DATA_DIR = os.path.join(REPO_DIR, "data")
train_dir = os.path.join(DATA_DIR, "train")

if not os.path.exists(train_dir) or len(glob.glob(os.path.join(train_dir, "*.parquet"))) < 660:
    dl = subprocess.Popen(
        ["bash", "scripts/download_data.sh", DATA_DIR],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in iter(dl.stdout.readline, b""):
        sys.stdout.write(line.decode(errors="replace")); sys.stdout.flush()
    dl.wait()
    if dl.returncode != 0:
        raise SystemExit("download_data.sh failed")

n_batches = len(glob.glob(os.path.join(train_dir, "*.parquet")))
print(f"{n_batches} batch files ready")
if n_batches < 660:
    raise SystemExit(f"Expected 660 batch files, found {n_batches} -- download incomplete")


In [ ]:
# -- Shared helpers: run a training stage, and patch a config's checkpoint --
def run_stage(script, config, watch_dir=None):
    """Runs `python -u <script> --config <config>` via subprocess, streaming
    stdout, while a background thread mirrors any *.ckpt files written under
    watch_dir to Drive as they appear (debounced on stable file size, same
    pattern as the reference notebook's checkpoint watcher).
    """
    _stop = threading.Event()
    _seen = set()

    def watcher():
        if watch_dir is None:
            return
        while not _stop.is_set():
            for path in glob.glob(os.path.join(watch_dir, "*.ckpt")):
                if path in _seen:
                    continue
                try:
                    s1 = os.path.getsize(path); time.sleep(5); s2 = os.path.getsize(path)
                    if s1 != s2 or s1 == 0:
                        continue
                except OSError:
                    continue
                drive_dst = os.path.join(DRIVE_CHECKPOINT_DIR, os.path.basename(watch_dir), os.path.basename(path))
                os.makedirs(os.path.dirname(drive_dst), exist_ok=True)
                shutil.copy2(path, drive_dst)
                _seen.add(path)
                print(f"=== CHECKPOINT SYNCED: {os.path.basename(path)} -> Drive ===", flush=True)
            _stop.wait(30)

    threading.Thread(target=watcher, daemon=True).start()

    proc = subprocess.Popen(
        ["python", "-u", script, "--config", config],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in iter(proc.stdout.readline, b""):
        sys.stdout.write(line.decode(errors="replace")); sys.stdout.flush()
    proc.wait()
    _stop.set()

    if proc.returncode != 0:
        raise SystemExit(f"{script} failed with code {proc.returncode}")
    print(f"=== {script} ({config}) COMPLETE ===")


def set_checkpoint(config_path, new_checkpoint):
    """Regex-patches a config's checkpoint: line in place -- same technique
    the reference notebook uses to patch a config before a training stage.
    """
    path = os.path.join(REPO_DIR, config_path)
    with open(path) as f:
        cfg = f.read()
    cfg = re.sub(r"checkpoint: .*", f"checkpoint: {new_checkpoint}", cfg, count=1)
    with open(path, "w") as f:
        f.write(cfg)


In [ ]:
# -- Experiment 2: NeutrinoJEPAPET (JEPA pre-training) --
run_stage("train/pretrain.py", "configs/pretrain.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/pretrain_jepa_v1"))

set_checkpoint("configs/probe.yaml", "checkpoints/pretrain_jepa_v1/epoch99.ckpt")
run_stage("train/probe.py", "configs/probe.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/probe_v1"))

set_checkpoint("configs/finetune.yaml", "checkpoints/pretrain_jepa_v1/epoch99.ckpt")
run_stage("train/finetune.py", "configs/finetune.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/finetune_v1"))


In [ ]:
# -- Experiment 1: NeutrinoPET (MAE pre-training) --
run_stage("train/pretrain_mae.py", "configs/pretrain_mae.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/pretrain_mae_v1"))

set_checkpoint("configs/probe.yaml", "checkpoints/pretrain_mae_v1/epoch99.ckpt")
run_stage("train/probe.py", "configs/probe.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/probe_v1"))

set_checkpoint("configs/finetune.yaml", "checkpoints/pretrain_mae_v1/epoch99.ckpt")
run_stage("train/finetune.py", "configs/finetune.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/finetune_v1"))


In [ ]:
# -- Experiment 3: PET+heads from scratch (no pre-training, no checkpoint) --
run_stage("train/finetune.py", "configs/scratch.yaml",
          watch_dir=os.path.join(REPO_DIR, "checkpoints/scratch_v1"))
print("=== All 3 experiments complete ===")
